In [ ]:
import pandas as pd
import yaml
import os
from pathlib import Path
import mne

ROOT = Path.cwd().resolve().parent
with open(ROOT/'config.yaml','r') as file:
    config = yaml.safe_load(file)

aauwss_path = ROOT/config['aauwss']['path']
label_audit = []
labels = ['Missing', 'Wake', 'N1', 'REM', 'N3', 'N2']
for file in os.listdir(aauwss_path/'annotations'):
    df = pd.read_excel(aauwss_path/'annotations'/file,usecols=['Sleep Stage','Epoch Number'])
    participant_id = file[:10]
    epoch_val_counts = df['Sleep Stage'].value_counts()
    epochs_per_label = {f"{l}_epochs":epoch_val_counts[l] if l in epoch_val_counts.index else 0 for l in labels}

    e4_bvp_path = aauwss_path /'empatica'/participant_id/'BVP.csv'
    if e4_bvp_path.exists():
        header = pd.read_csv(e4_bvp_path, nrows=2, header=None)
        start_time_utc = float(header.iloc[0, 0])
        sample_rate_hz = float(header.iloc[1, 0])
        df_bvp = pd.read_csv(e4_bvp_path, skiprows=2, header=None)
        n_rows = len(df_bvp)
        duration_seconds = n_rows/sample_rate_hz
    else:
        n_rows = 0
        duration_seconds = 0.0
    
    label_audit.append({'participant':participant_id,
                        'n_rows': n_rows,
                        'duration': duration_seconds, 
                        **epochs_per_label})


label_audit_df = pd.DataFrame(label_audit).rename(columns={'Wake_epochs':'W_epochs', 'REM_epochs':'R_epochs'})
output_dir = ROOT/'results'/'inventory'
label_audit_df.to_csv(output_dir/'aauwss_label_audit.csv',index=False)
print(f"Saved aauwss_label_audit to {output_dir}.")


In [ ]:
edf_dir = aauwss_path/'edfs'
edf_files = sorted([f for f in os.listdir(edf_dir) if f.endswith('.edf')])
if edf_files:
    sample_edf = edf_dir / edf_files[0]
    raw = mne.io.read_raw_edf(sample_edf, verbose=False)
    
    psg_channels_info = []
    for ch_name in raw.ch_names:
        psg_channels_info.append({'channel_name': ch_name, 'sampling_rate_hz': raw.info['sfreq']})

df_psg = pd.DataFrame(psg_channels_info)
df_psg.to_csv(output_dir / 'aauwss_psg_channels.csv', index=False)

In [ ]:
import os
from pathlib import Path
import yaml
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve().parent if Path.cwd().name != 'project' else Path.cwd().resolve()
with open(ROOT / 'config.yaml', 'r') as file:
    config = yaml.safe_load(file)

aauwss_path = ROOT / config['aauwss']['path']
output_dir = ROOT / 'results' / 'inventory'
output_dir.mkdir(parents=True, exist_ok=True)

EXPECTED_RATES = {'BVP': 64.0,'ACC': 32.0,'EDA': 4.0,'TEMP': 4.0}

all_alignments = []
annotation_files = sorted([f for f in os.listdir(aauwss_path / 'annotations') if f.endswith('.xlsx')])

for file in annotation_files:
    participant_id = file[:10]  # Extracts participant ID e.g., 'sub-001'
    df_annot = pd.read_excel(aauwss_path / 'annotations' / file)
    empatica_dir = aauwss_path / 'empatica' / participant_id

    sensor_segments = {}
    for sensor in ['BVP', 'ACC', 'EDA', 'TEMP']:
        sensor_files = sorted(list(empatica_dir.glob(f'**/{sensor}.csv')))
        segments = []
        for s_path in sensor_files:
            header = pd.read_csv(s_path, nrows=2, header=None)
            start_ts = float(header.iloc[0, 0])
            srate = float(header.iloc[1, 0])
            
            df_seg = pd.read_csv(s_path, skiprows=2, header=None)
            n_rows = len(df_seg)
            duration_sec = n_rows / srate if srate > 0 else 0.0
            
            start_dt = pd.to_datetime(start_ts, unit='s', utc=True)
            end_dt = start_dt + pd.Timedelta(seconds=duration_sec)
            
            segments.append({'path': s_path,'start_dt': start_dt,'end_dt': end_dt,'n_rows': n_rows,'srate': 
                             srate,'start_ts': start_ts})
        sensor_segments[sensor] = sorted(segments, key=lambda x: x['start_dt'])

    for idx, row in df_annot.iterrows():
        stage = str(row['Sleep Stage']).strip()
        epoch_num = row['Epoch Number']
        raw_time_val = row['Event Start Time']
        
        local_dt = pd.to_datetime(str(raw_time_val))
        epoch_start_utc = local_dt.tz_localize('Europe/Copenhagen', ambiguous='NaT', nonexistent='NaT').tz_convert('UTC')
        epoch_end_utc = epoch_start_utc + pd.Timedelta(seconds=30)
        
        is_usable = stage not in ['Missing', 'P', 'nan', 'None']
        reason = 'OK' if is_usable else f'Stage_{stage}'

        bvp_segs = sensor_segments.get('BVP', [])
        matching_bvp = None
        for seg in bvp_segs:
            if seg['start_dt'] <= epoch_start_utc and epoch_end_utc <= seg['end_dt']:
                matching_bvp = seg
                break
        
        if matching_bvp is not None:
            record = {'participant': participant_id,'epoch_number': epoch_num,'sleep_stage': stage,'epoch_start_utc': epoch_start_utc,
                      'usable': is_usable,'reason': reason}
            
            for sensor in ['BVP', 'ACC', 'EDA', 'TEMP']:
                segs = sensor_segments.get(sensor, [])
                match_seg = None
                for seg in segs:
                    if seg['start_dt'] <= epoch_start_utc and epoch_end_utc <= seg['end_dt']:
                        match_seg = seg
                        break
                
                if match_seg is not None:
                    offset_sec = (epoch_start_utc - match_seg['start_dt']).total_seconds()
                    srate = match_seg['srate']
                    start_row = int(round(offset_sec * srate))
                    end_row = start_row + int(round(srate * 30))
                    
                    record[f'{sensor.lower()}_start_row'] = start_row
                    record[f'{sensor.lower()}_end_row'] = end_row
                    record[f'{sensor.lower()}_srate'] = srate
                else:
                    record[f'{sensor.lower()}_start_row'] = np.nan
                    record[f'{sensor.lower()}_end_row'] = np.nan
                    record[f'{sensor.lower()}_srate'] = EXPECTED_RATES[sensor]
                    record['usable'] = False
                    record['reason'] = f'Missing_{sensor}_Segment'
                    
            all_alignments.append(record)
        else:
            all_alignments.append({'participant': participant_id,'epoch_number': epoch_num,'sleep_stage': 'Missing','epoch_start_utc': epoch_start_utc,
                                   'bvp_start_row': np.nan, 'bvp_end_row': np.nan, 'bvp_srate': 64.0,'acc_start_row': np.nan, 
                                   'acc_end_row': np.nan, 'acc_srate': 32.0,'eda_start_row': np.nan, 'eda_end_row': np.nan, 
                                   'eda_srate': 4.0,'temp_start_row': np.nan, 'temp_end_row': np.nan, 'temp_srate': 4.0,'usable': False,
                                   'reason': 'Gap_or_Restart_Dropout'})

df_alignment = pd.DataFrame(all_alignments)
df_alignment.to_csv(output_dir / 'aauwss_epoch_alignment.csv', index=False)

missing_summary = df_alignment.groupby(['participant', 'usable']).size().unstack(fill_value=0)
print(f"AAUWSS Epoch Alignment Manifest saved to: {output_dir / 'aauwss_epoch_alignment.csv'}\n")
print("Usable vs Excluded Epochs per Participant:")
print(missing_summary)

AAUWSS Epoch Alignment Manifest saved to: D:\Adobe\projects\research proj\results\inventory\aauwss_epoch_alignment.csv

Usable vs Excluded Epochs per Participant:
usable       False  True 
participant              
subject_01       0    845
subject_02       0    751
subject_03       2    929
subject_04       0    920
subject_05       0    836
subject_06       0    778
subject_07       0    929
subject_08       0    875
subject_09       0    920
subject_10       1    946
subject_11      14    905
subject_12     328    573
subject_13      20    888


In [6]:
import numpy as np

def check_epoch_quality(epoch_signals, label, thresholds):
    if label not in ['R','REM','N1','N2','N3','W','Wake']: return (False, f'Label is {label}')

    for k, signal in epoch_signals.items():
        signal_arr = np.asarray(signal, dtype=float)
        if len(signal_arr) == 0:
            return (False, f'{k} is empty')
        missing_ratio = np.isnan(signal_arr).mean()
        if missing_ratio > thresholds['max_missing_ratio']:
            return (False, f'{k} has low coverage({round(missing_ratio * 100, 1)}%)')

    temp = np.asarray(epoch_signals['TEMP'], dtype=float)
    temp_valid = temp[~np.isnan(temp)]
    if len(temp_valid) > 0:
        temp_off_wrist_ratio = ((temp_valid < thresholds['temp_min']) | (temp_valid > thresholds['temp_max'])).mean()
        if temp_off_wrist_ratio > thresholds['temp_off_wrist_max_ratio']:
            return (False, 'Band not on wrist(TEMP off bounds)')

    eda = np.asarray(epoch_signals['EDA'], dtype=float)
    eda_valid = eda[~np.isnan(eda)]
    if len(eda_valid) > 0:
        if (eda_valid < thresholds['eda_min']).all():
            return (False, 'Band not on wrist(EDA too low)')
    
    bvp = np.asarray(epoch_signals['BVP'], dtype=float)
    bvp_valid = bvp[~np.isnan(bvp)]
    if len(bvp_valid) > 0 and np.var(bvp_valid) < thresholds['bvp_min_var']:
        return (False, 'Flat BVP')
    
    hr = np.asarray(epoch_signals['HR'], dtype=float)
    hr_valid = hr[~np.isnan(hr)]
    if len(hr_valid) > 0:
        hr_out_of_bounds = ((hr_valid < thresholds['hr_min']) | (hr_valid > thresholds['hr_max'])).mean()
        if hr_out_of_bounds > 0.0:
            return (False, 'HR out of bounds')

    return (True, 'OK')

In [13]:
import pandas as pd
import yaml
import os
from pathlib import Path

ROOT = Path.cwd().resolve().parent
with open(ROOT / 'config.yaml', 'r') as file:
    config = yaml.safe_load(file)

dreamt_data_path = ROOT / config['dreamt']['path'] / 'data_64Hz'
dreamt_epoch_quality_log = []

for file in os.listdir(dreamt_data_path):
    participant_id = file[:4]
    file_path = dreamt_data_path/file
    df_participant = pd.read_csv(file_path)
    
    non_p_indices = df_participant.index[df_participant['Sleep_Stage'] != 'P'].tolist()
    if not non_p_indices:
        continue
    first_scored_row = non_p_indices[0]
    total_rows = len(df_participant)
    
    epoch_num = 0
    for start_row in range(first_scored_row, total_rows-config['dreamt']['rows_per_epoch']+1, config['dreamt']['rows_per_epoch']):
        end_row = start_row + config['dreamt']['rows_per_epoch']
        epoch_df = df_participant.iloc[start_row:end_row]
        
        epoch_signals = {'BVP': epoch_df['BVP'].to_numpy(),'ACC_X': epoch_df['ACC_X'].to_numpy(),
                         'ACC_Y': epoch_df['ACC_Y'].to_numpy(),'ACC_Z': epoch_df['ACC_Z'].to_numpy(),
                         'EDA': epoch_df['EDA'].to_numpy(),'TEMP': epoch_df['TEMP'].to_numpy(),
                         'HR': epoch_df['HR'].to_numpy()}
        
        label = epoch_df['Sleep_Stage'].iloc[0]
        usable, reason = check_epoch_quality(epoch_signals, label, config['thresholds'])
        
        dreamt_epoch_quality_log.append({'participant': participant_id,'epoch_number': epoch_num,'sleep_stage': label,
                                         'start_row': start_row,'end_row': end_row,'usable': usable,'reason': reason})
        epoch_num += 1


output_dir = ROOT / 'results' / 'inventory'
df_dreamt_quality = pd.DataFrame(dreamt_epoch_quality_log)
df_dreamt_quality.to_csv(output_dir / 'dreamt_quality_audit.csv', index=False)
print(f"DREAMT: Processed {len(df_dreamt_quality)} epochs across all participants.")

DREAMT: Processed 80252 epochs across all participants.


In [7]:
from pathlib import Path
import pandas as pd
import numpy as np
import yaml

ROOT = Path.cwd().resolve().parent
with open(ROOT / 'config.yaml', 'r') as file:
    config = yaml.safe_load(file)
aauwss_path = ROOT / config['aauwss']['path']
output_dir = ROOT / 'results' / 'inventory'
alignment_csv = output_dir / 'aauwss_epoch_alignment.csv'
df_alignment = pd.read_csv(alignment_csv)
aauwss_epoch_quality_log = []

def load_segment_files(seg_dir):
    files = ['BVP.csv', 'ACC.csv', 'EDA.csv', 'TEMP.csv', 'HR.csv']
    data = {}
    for f in files:
        fp = seg_dir / f
        if fp.exists():
            data[f.replace('.csv', '')] = pd.read_csv(fp, skiprows=2, header=None)[0].to_numpy()
        else:
            data[f.replace('.csv', '')] = np.array([])
    return data

grouped = df_alignment.groupby('participant')
for participant_id, group in grouped:
    empatica_dir = aauwss_path / 'empatica' / participant_id
    current_seg_dir = None
    cached_signals = {}
    
    for _, epoch_row in group.iterrows():
        if pd.isna(epoch_row['segment_file']) or not epoch_row['usable']:
            aauwss_epoch_quality_log.append({
                'participant': participant_id, 'epoch_number': epoch_row['epoch_number'],
                'sleep_stage': epoch_row['sleep_stage'], 'usable': False,
                'reason': str(epoch_row['reason'])
            })
            continue

        segment_file = epoch_row['segment_file']    
        segment_dirs = list(empatica_dir.glob(f"**/{segment_file}"))
        if not segment_dirs:
            continue
        seg_dir = segment_dirs[0].parent
       
        if seg_dir != current_seg_dir:
            cached_signals = load_segment_files(seg_dir)
            current_seg_dir = seg_dir
        start_64hz = int(epoch_row['start_row_64hz']) if not pd.isna(epoch_row['start_row_64hz']) else 0
        
        s_bvp  = max(0, start_64hz - 2)
        s_acc  = max(0, int(start_64hz / 2) - 2)
        s_eda  = max(0, int(start_64hz / 16) - 2)
        s_hr   = max(0, int(start_64hz / 64) - 2)
        epoch_signals = {'BVP':  cached_signals['BVP'][s_bvp:s_bvp + 1920] if len(cached_signals['BVP']) > s_bvp else np.array([]),
                         'ACC_X': cached_signals['ACC'][s_acc:s_acc + 960] if len(cached_signals['ACC']) > s_acc else np.array([]),
                         'EDA':  cached_signals['EDA'][s_eda:s_eda + 120] if len(cached_signals['EDA']) > s_eda else np.array([]),
                         'TEMP': cached_signals['TEMP'][s_eda:s_eda + 120] if len(cached_signals['TEMP']) > s_eda else np.array([]),
                         'HR': cached_signals['HR'][s_hr:s_hr + 30] if len(cached_signals['HR']) > s_hr else np.array([])}

        usable, reason = check_epoch_quality(epoch_signals, epoch_row['sleep_stage'], config['thresholds'])
        aauwss_epoch_quality_log.append({'participant': participant_id, 'epoch_number': epoch_row['epoch_number'],
                                         'sleep_stage': epoch_row['sleep_stage'], 'usable': usable, 'reason': reason})

df_aauwss_quality = pd.DataFrame(aauwss_epoch_quality_log)
df_aauwss_quality.to_csv(output_dir / 'aauwss_quality_audit.csv', index=False)
print(f"AAUWSS: Processed {len(df_aauwss_quality)} epochs across all participants.")

AAUWSS: Processed 11460 epochs across all participants.
